# Đọc dữ liệu
- Nhiệm vụ chính là đọc dữ liệu từ file csv sang parquet bằng Spark.
- Vì dữ liệu ban đầu rất lớn, mục tiêu của chúng ta là sẽ chỉ lấy ra những phần quan trong, lọc và loại bỏ những cột không quan trọng.

### Buoc 1: Khoi tao SparkSession

Muc tieu cua buoc nay:
- Tao SparkSession de lam viec voi du lieu lon bang PySpark.
- Chay Spark o che do local tren may ca nhan.
- Khai bao san cac duong dan input/output cho cac buoc tiep theo.
- Kiem tra nhanh Spark version va duong dan file CSV.

#### Viec can lam trong buoc 1

1. Kiem tra Python environment da co pyspark hay chua.
2. Kiem tra may da cai Java va Java version co phu hop voi Spark hay chua.
3. Tao SparkSession bang SparkSession.builder.
4. Khai bao cac duong dan input/output dung cho cac buoc tiep theo.
5. In thong tin Spark de xac nhan khoi tao thanh cong.

In [1]:
import importlib.util
import re
import shutil
import subprocess


SUPPORTED_JAVA_MAJOR_VERSIONS = {17, 21}

if importlib.util.find_spec("pyspark") is None:
    raise ModuleNotFoundError(
        "Chua cai pyspark trong kernel hien tai. "
        "Hay chay: python -m pip install pyspark"
    )

java_path = shutil.which("java")
if java_path is None:
    raise RuntimeError(
        "Chua tim thay Java trong PATH. "
        "Hay cai JDK va mo lai VS Code/Jupyter de PATH duoc cap nhat."
    )

java_check = subprocess.run(
    [java_path, "-version"],
    capture_output=True,
    text=True,
    check=False,
)
java_version_text = java_check.stderr or java_check.stdout
first_java_line = java_version_text.splitlines()[0] if java_version_text else "unknown"
java_version_match = re.search(r'\"(\d+)', java_version_text)

if java_version_match is None:
    raise RuntimeError(f"Khong doc duoc Java version tu: {first_java_line}")

java_major_version = int(java_version_match.group(1))
if java_major_version not in SUPPORTED_JAVA_MAJOR_VERSIONS:
    raise RuntimeError(
        f"Java hien tai la version {java_major_version}, chua phu hop voi Spark. "
        "Hay dung JDK 17 hoac JDK 21, sau do cap nhat JAVA_HOME/PATH va restart VS Code/Jupyter."
    )

print("OK: pyspark da san sang")
print("OK: Java version phu hop voi Spark")
print("Java path:", java_path)
print("Java version:", first_java_line)


OK: pyspark da san sang
OK: Java version phu hop voi Spark
Java path: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
Java version: java version "21.0.11" 2026-04-21 LTS


In [2]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession


# Dam bao Spark driver va Python worker dung cung mot Python executable.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Notebook co the duoc chay tu thu muc goc project hoac tu week_3.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "week_3" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_pyspark"
PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"

TRAIN_CSV_PATH = DATA_DIR / "train_v2.csv"
TEST_CSV_PATH = DATA_DIR / "test_v2.csv"

TRAIN_PARQUET_PATH = PARQUET_DIR / "train_sessions"
TEST_PARQUET_PATH = PARQUET_DIR / "test_sessions"
LOG_PATH = PARQUET_DIR / "read_csv_to_parquet_log.json"

spark = (
    SparkSession.builder
    .appName("GA Customer Revenue - CSV to Parquet") # 
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Python executable:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Train CSV exists:", TRAIN_CSV_PATH.exists())
print("Test CSV exists:", TEST_CSV_PATH.exists())


Spark version: 4.1.2
Spark master: local[*]
Python executable: f:\ide\anaconda\python.exe
Project root: g:\ds
Train CSV exists: True
Test CSV exists: True


### Buoc 2: Doc thu header va schema CSV

Viec can lam trong buoc 2:
1. Khai bao option doc CSV phu hop voi file co cot JSON dang string.
2. Doc header cua train/test bang Spark ma khong scan toan bo file.
3. In so luong cot va danh sach cot.
4. In schema mac dinh cua Spark khi chua parse/cast kieu du lieu.
5. So sanh cot train va test de biet co cot nao chi xuat hien o mot file hay khong.

In [3]:
CSV_READ_OPTIONS = {
    "header": "true",
    "quote": '"',
    "escape": '"',
    "multiLine": "false",
    "mode": "PERMISSIVE",
}

print(f"Train CSV size: {TRAIN_CSV_PATH.stat().st_size / 1024**3:.2f} GB")
print(f"Test CSV size: {TEST_CSV_PATH.stat().st_size / 1024**3:.2f} GB")

train_header_df = (
    spark.read
    .options(**CSV_READ_OPTIONS)
    .csv(str(TRAIN_CSV_PATH))
    .limit(0)
)

test_header_df = (
    spark.read
    .options(**CSV_READ_OPTIONS)
    .csv(str(TEST_CSV_PATH))
    .limit(0)
)


def show_header_and_schema(name, df):
    print(f"\n{name} column count:", len(df.columns))
    print(f"{name} columns:")
    for index, column_name in enumerate(df.columns, start=1):
        print(f"{index:02d}. {column_name}")
    print(f"\n{name} schema:")
    df.printSchema()


show_header_and_schema("train_v2", train_header_df)
show_header_and_schema("test_v2", test_header_df)

train_only_columns = sorted(set(train_header_df.columns) - set(test_header_df.columns))
test_only_columns = sorted(set(test_header_df.columns) - set(train_header_df.columns))

print("\nColumns only in train:", train_only_columns)
print("Columns only in test:", test_only_columns)


Train CSV size: 23.67 GB
Test CSV size: 7.09 GB

train_v2 column count: 13
train_v2 columns:
01. channelGrouping
02. customDimensions
03. date
04. device
05. fullVisitorId
06. geoNetwork
07. hits
08. socialEngagementType
09. totals
10. trafficSource
11. visitId
12. visitNumber
13. visitStartTime

train_v2 schema:
root
 |-- channelGrouping: string (nullable = true)
 |-- customDimensions: string (nullable = true)
 |-- date: string (nullable = true)
 |-- device: string (nullable = true)
 |-- fullVisitorId: string (nullable = true)
 |-- geoNetwork: string (nullable = true)
 |-- hits: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- totals: string (nullable = true)
 |-- trafficSource: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)


test_v2 column count: 13
test_v2 columns:
01. channelGrouping
02. customDimensions
03. date
04. device
05. fullVisitorId
06

### Buoc 3: Doc sample nho

Viec can lam trong buoc 3:
1. Doc mot so dong nho tu train/test de inspect du lieu that.
2. Cache sample tam thoi vi ta se xem nhieu lan trong notebook.
3. Kiem tra so dong sample doc duoc.
4. Xem nhanh cac cot session-level de hieu grain cua du lieu.
5. Xem preview cac cot JSON dang string de chuan bi cho buoc parse JSON tiep theo.

In [4]:
from pyspark.sql import functions as F


SAMPLE_ROW_COUNT = 100
PREVIEW_ROW_COUNT = 5
PREVIEW_TEXT_LENGTH = 180


def read_csv_sample(csv_path, row_count=SAMPLE_ROW_COUNT):
    full_df = spark.read.options(**CSV_READ_OPTIONS).csv(str(csv_path))
    sample_rows = full_df.take(row_count)
    sample_df = spark.createDataFrame(sample_rows, schema=full_df.schema).cache()
    return sample_df


train_sample_df = read_csv_sample(TRAIN_CSV_PATH)
test_sample_df = read_csv_sample(TEST_CSV_PATH)

train_sample_count = train_sample_df.count()
test_sample_count = test_sample_df.count()

print("Train sample rows:", train_sample_count)
print("Test sample rows:", test_sample_count)

session_preview_columns = [
    "date",
    "fullVisitorId",
    "visitId",
    "visitNumber",
    "visitStartTime",
    "channelGrouping",
    "socialEngagementType",
]

print("\nTrain session-level preview:")
train_sample_df.select(*session_preview_columns).show(PREVIEW_ROW_COUNT, truncate=80)

print("\nTest session-level preview:")
test_sample_df.select(*session_preview_columns).show(PREVIEW_ROW_COUNT, truncate=80)

json_columns = ["customDimensions", "device", "geoNetwork", "hits", "totals", "trafficSource"]


def show_json_preview(name, df):
    preview_expressions = [
        F.substring(F.col(column_name), 1, PREVIEW_TEXT_LENGTH).alias(f"{column_name}_preview")
        for column_name in json_columns
    ]

    print(f"\n{name} JSON/string column preview:")
    df.select("fullVisitorId", "visitId", *preview_expressions).show(
        PREVIEW_ROW_COUNT,
        truncate=120,
    )


show_json_preview("Train", train_sample_df)
show_json_preview("Test", test_sample_df)


Train sample rows: 100
Test sample rows: 100

Train session-level preview:
+--------+-------------------+----------+-----------+--------------+---------------+--------------------+
|    date|      fullVisitorId|   visitId|visitNumber|visitStartTime|channelGrouping|socialEngagementType|
+--------+-------------------+----------+-----------+--------------+---------------+--------------------+
|20171016|3162355547410993243|1508198450|          1|    1508198450| Organic Search|Not Socially Engaged|
|20171016|8934116514970143966|1508176307|          6|    1508176307|       Referral|Not Socially Engaged|
|20171016|7992466427990357681|1508201613|          1|    1508201613|         Direct|Not Socially Engaged|
|20171016|9075655783635761930|1508169851|          1|    1508169851| Organic Search|Not Socially Engaged|
|20171016|6960673291025684308|1508190552|          1|    1508190552| Organic Search|Not Socially Engaged|
+--------+-------------------+----------+-----------+--------------+---------

### Buoc 4: Chon cot can giu o muc session-level

Viec can lam trong buoc 4:
1. Xac dinh cac cot dinh danh session va visitor.
2. Giu cac cot context don gian nhu ngay, kenh truy cap, social engagement.
3. Giu cac cot JSON quan trong de parse thanh feature o cac buoc sau.
4. Tam thoi loai cot `hits` vi cot nay rat nang va nam o muc hit-level, khong phai session-level truc tiep.
5. Tao ham select cot session-level de tai su dung cho sample va full data.

In [5]:
SESSION_ID_COLUMNS = [
    "fullVisitorId",
    "visitId",
    "visitNumber",
    "visitStartTime",
]

SESSION_TIME_COLUMNS = [
    "date",
]

SESSION_CONTEXT_COLUMNS = [
    "channelGrouping",
    "socialEngagementType",
]

SESSION_JSON_COLUMNS = [
    "totals",
    "device",
    "geoNetwork",
    "trafficSource",
    "customDimensions",
]

SESSION_LEVEL_COLUMNS = (
    SESSION_ID_COLUMNS
    + SESSION_TIME_COLUMNS
    + SESSION_CONTEXT_COLUMNS
    + SESSION_JSON_COLUMNS
)


def validate_columns_exist(df, required_columns):
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")


def select_session_level_columns(df):
    validate_columns_exist(df, SESSION_LEVEL_COLUMNS)
    return df.select(*SESSION_LEVEL_COLUMNS)


train_session_sample_df = select_session_level_columns(train_sample_df).cache()
test_session_sample_df = select_session_level_columns(test_sample_df).cache()

dropped_columns = sorted(set(train_sample_df.columns) - set(SESSION_LEVEL_COLUMNS))

print("Session-level columns to keep:")
for index, column_name in enumerate(SESSION_LEVEL_COLUMNS, start=1):
    print(f"{index:02d}. {column_name}")

print("\nDropped columns at this step:", dropped_columns)
print("Train session sample rows:", train_session_sample_df.count())
print("Test session sample rows:", test_session_sample_df.count())

print("\nTrain session-level selected schema:")
train_session_sample_df.printSchema()

print("\nTrain session-level selected preview:")
train_session_sample_df.select(
    "fullVisitorId",
    "visitId",
    "date",
    "channelGrouping",
    "totals",
).show(PREVIEW_ROW_COUNT, truncate=120)


Session-level columns to keep:
01. fullVisitorId
02. visitId
03. visitNumber
04. visitStartTime
05. date
06. channelGrouping
07. socialEngagementType
08. totals
09. device
10. geoNetwork
11. trafficSource
12. customDimensions

Dropped columns at this step: ['hits']
Train session sample rows: 100
Test session sample rows: 100

Train session-level selected schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- totals: string (nullable = true)
 |-- device: string (nullable = true)
 |-- geoNetwork: string (nullable = true)
 |-- trafficSource: string (nullable = true)
 |-- customDimensions: string (nullable = true)


Train session-level selected preview:
+-------------------+----------+--------+---------------+--------------

### Buoc 5: Parse cac cot JSON quan trong

Viec can lam trong buoc 5:
1. Khai bao schema cho cac cot JSON session-level: `totals`, `device`, `geoNetwork`, `trafficSource`.
2. Dung `from_json` de parse string JSON thanh struct.
3. Tao cac cot feature tam thoi co prefix ro rang: `totals_*`, `device_*`, `geo_*`, `traffic_*`.
4. Trich `customDimensions.value` bang regex vi cot nay dang o dang Python-like string, khong phai JSON chuan.
5. Giu lai cot JSON goc trong buoc nay; viec drop cot goc se lam o buoc 9.

In [6]:
from pyspark.sql.types import BooleanType, StringType, StructField, StructType


totals_schema = StructType([
    StructField("visits", StringType(), True),
    StructField("hits", StringType(), True),
    StructField("pageviews", StringType(), True),
    StructField("bounces", StringType(), True),
    StructField("newVisits", StringType(), True),
    StructField("timeOnSite", StringType(), True),
    StructField("sessionQualityDim", StringType(), True),
    StructField("transactions", StringType(), True),
    StructField("transactionRevenue", StringType(), True),
    StructField("totalTransactionRevenue", StringType(), True),
])

device_schema = StructType([
    StructField("browser", StringType(), True),
    StructField("operatingSystem", StringType(), True),
    StructField("isMobile", BooleanType(), True),
    StructField("deviceCategory", StringType(), True),
])

geo_network_schema = StructType([
    StructField("continent", StringType(), True),
    StructField("subContinent", StringType(), True),
    StructField("country", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metro", StringType(), True),
    StructField("city", StringType(), True),
    StructField("networkDomain", StringType(), True),
])

adwords_click_info_schema = StructType([
    StructField("campaignId", StringType(), True),
    StructField("adGroupId", StringType(), True),
    StructField("creativeId", StringType(), True),
    StructField("criteriaId", StringType(), True),
    StructField("page", StringType(), True),
    StructField("slot", StringType(), True),
    StructField("gclId", StringType(), True),
    StructField("isVideoAd", BooleanType(), True),
])

traffic_source_schema = StructType([
    StructField("campaign", StringType(), True),
    StructField("source", StringType(), True),
    StructField("medium", StringType(), True),
    StructField("keyword", StringType(), True),
    StructField("referralPath", StringType(), True),
    StructField("adContent", StringType(), True),
    StructField("isTrueDirect", BooleanType(), True),
    StructField("adwordsClickInfo", adwords_click_info_schema, True),
])


def parse_session_json_columns(df):
    parsed_df = (
        df
        .withColumn("totals_struct", F.from_json(F.col("totals"), totals_schema))
        .withColumn("device_struct", F.from_json(F.col("device"), device_schema))
        .withColumn("geo_struct", F.from_json(F.col("geoNetwork"), geo_network_schema))
        .withColumn("traffic_struct", F.from_json(F.col("trafficSource"), traffic_source_schema))
    )

    return parsed_df.select(
        "*",
        F.col("totals_struct.visits").alias("totals_visits_raw"),
        F.col("totals_struct.hits").alias("totals_hits_raw"),
        F.col("totals_struct.pageviews").alias("totals_pageviews_raw"),
        F.col("totals_struct.bounces").alias("totals_bounces_raw"),
        F.col("totals_struct.newVisits").alias("totals_new_visits_raw"),
        F.col("totals_struct.timeOnSite").alias("totals_time_on_site_raw"),
        F.col("totals_struct.sessionQualityDim").alias("totals_session_quality_dim_raw"),
        F.col("totals_struct.transactions").alias("totals_transactions_raw"),
        F.col("totals_struct.transactionRevenue").alias("totals_transaction_revenue_raw"),
        F.col("totals_struct.totalTransactionRevenue").alias("totals_total_transaction_revenue_raw"),
        F.col("device_struct.browser").alias("device_browser"),
        F.col("device_struct.operatingSystem").alias("device_operating_system"),
        F.col("device_struct.isMobile").alias("device_is_mobile"),
        F.col("device_struct.deviceCategory").alias("device_category"),
        F.col("geo_struct.continent").alias("geo_continent"),
        F.col("geo_struct.subContinent").alias("geo_sub_continent"),
        F.col("geo_struct.country").alias("geo_country"),
        F.col("geo_struct.region").alias("geo_region"),
        F.col("geo_struct.metro").alias("geo_metro"),
        F.col("geo_struct.city").alias("geo_city"),
        F.col("geo_struct.networkDomain").alias("geo_network_domain"),
        F.col("traffic_struct.campaign").alias("traffic_campaign"),
        F.col("traffic_struct.source").alias("traffic_source"),
        F.col("traffic_struct.medium").alias("traffic_medium"),
        F.col("traffic_struct.keyword").alias("traffic_keyword"),
        F.col("traffic_struct.referralPath").alias("traffic_referral_path"),
        F.col("traffic_struct.adContent").alias("traffic_ad_content"),
        F.col("traffic_struct.isTrueDirect").alias("traffic_is_true_direct"),
        F.col("traffic_struct.adwordsClickInfo.gclId").alias("traffic_gcl_id"),
        F.regexp_extract(F.col("customDimensions"), r"'value': '([^']+)'", 1).alias("custom_dimension_value"),
    )


train_parsed_sample_df = parse_session_json_columns(train_session_sample_df).cache()
test_parsed_sample_df = parse_session_json_columns(test_session_sample_df).cache()

parsed_preview_columns = [
    "fullVisitorId",
    "visitId",
    "date",
    "totals_hits_raw",
    "totals_pageviews_raw",
    "totals_transaction_revenue_raw",
    "device_browser",
    "device_category",
    "geo_country",
    "traffic_source",
    "traffic_medium",
    "custom_dimension_value",
]

print("Parsed train sample rows:", train_parsed_sample_df.count())
print("Parsed test sample rows:", test_parsed_sample_df.count())

print("\nParsed columns added:")
for column_name in parsed_preview_columns[3:]:
    print("-", column_name)

print("\nParsed train sample preview:")
train_parsed_sample_df.select(*parsed_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nParsed train sample schema preview:")
train_parsed_sample_df.select(*parsed_preview_columns).printSchema()


Parsed train sample rows: 100
Parsed test sample rows: 100

Parsed columns added:
- totals_hits_raw
- totals_pageviews_raw
- totals_transaction_revenue_raw
- device_browser
- device_category
- geo_country
- traffic_source
- traffic_medium
- custom_dimension_value

Parsed train sample preview:
+-------------------+----------+--------+---------------+--------------------+------------------------------+--------------+---------------+-------------+----------------+--------------+----------------------+
|      fullVisitorId|   visitId|    date|totals_hits_raw|totals_pageviews_raw|totals_transaction_revenue_raw|device_browser|device_category|  geo_country|  traffic_source|traffic_medium|custom_dimension_value|
+-------------------+----------+--------+---------------+--------------------+------------------------------+--------------+---------------+-------------+----------------+--------------+----------------------+
|3162355547410993243|1508198450|20171016|              1|                   

### Buoc 6: Tao cac cot numeric sach

Viec can lam trong buoc 6:
1. Chuyen cac cot id/thoi gian tu string sang kieu so hoac date/timestamp phu hop.
2. Chuyen cac chi so trong `totals_*_raw` sang integer/long.
3. Thay null bang 0 cho cac metric dang dem nhu hits, pageviews, transactions, revenue.
4. Tao revenue dang `long` theo micros va revenue dang `double` theo don vi tien te de de doc hon.
5. Kiem tra schema va preview de dam bao cac cot numeric da sach.

In [ ]:
REVENUE_MICRO_DIVISOR = 1_000_000.0


def clean_int_column(column_name, default_value=0):
    return F.coalesce(F.col(column_name).cast("int"), F.lit(default_value))


def clean_long_column(column_name, default_value=0):
    return F.coalesce(F.col(column_name).cast("long"), F.lit(default_value))


def create_clean_numeric_columns(df):
    cleaned_df = (
        df
        .withColumn("visit_id", F.col("visitId").cast("long"))
        .withColumn("visit_number", F.col("visitNumber").cast("int"))
        .withColumn("visit_start_time", F.col("visitStartTime").cast("long"))
        .withColumn("visit_start_timestamp", F.to_timestamp(F.from_unixtime(F.col("visitStartTime").cast("long"))))
        .withColumn("session_date", F.to_date(F.col("date"), "yyyyMMdd"))
        .withColumn("totals_visits", clean_int_column("totals_visits_raw"))
        .withColumn("totals_hits", clean_int_column("totals_hits_raw"))
        .withColumn("totals_pageviews", clean_int_column("totals_pageviews_raw"))
        .withColumn("totals_bounces", clean_int_column("totals_bounces_raw"))
        .withColumn("totals_new_visits", clean_int_column("totals_new_visits_raw"))
        .withColumn("totals_time_on_site", clean_int_column("totals_time_on_site_raw"))
        .withColumn("totals_session_quality_dim", clean_int_column("totals_session_quality_dim_raw"))
        .withColumn("totals_transactions", clean_int_column("totals_transactions_raw"))
        .withColumn("totals_transaction_revenue", clean_long_column("totals_transaction_revenue_raw"))
        .withColumn("totals_total_transaction_revenue", clean_long_column("totals_total_transaction_revenue_raw"))
    )

    return (
        cleaned_df
        .withColumn("transaction_revenue", F.col("totals_transaction_revenue") / F.lit(REVENUE_MICRO_DIVISOR))
        .withColumn("total_transaction_revenue", F.col("totals_total_transaction_revenue") / F.lit(REVENUE_MICRO_DIVISOR))
    )


train_numeric_sample_df = create_clean_numeric_columns(train_parsed_sample_df).cache()
test_numeric_sample_df = create_clean_numeric_columns(test_parsed_sample_df).cache()

numeric_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "visit_number",
    "session_date",
    "visit_start_timestamp",
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_transactions",
    "totals_total_transaction_revenue",
    "total_transaction_revenue",
]

print("Train numeric sample rows:", train_numeric_sample_df.count())
print("Test numeric sample rows:", test_numeric_sample_df.count())

print("\nNumeric columns preview:")
train_numeric_sample_df.select(*numeric_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nNumeric columns schema:")
train_numeric_sample_df.select(*numeric_preview_columns).printSchema()

print("\nQuick numeric summary:")
train_numeric_sample_df.select(
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_transactions",
    "total_transaction_revenue",
).summary("count", "min", "mean", "max").show(truncate=False)
